In [2]:
# Imports
!pip install -q hopsworks pandas requests numpy matplotlib seaborn confluent-kafka xgboost tensorflow shap
import os
from dotenv import load_dotenv
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import hopsworks
from google.colab import userdata
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, GRU
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler

CITY = "Lahore"
DAYS_TO_FETCH    = 700
HOPSWORKS_API_KEY = userdata.get("hopscotch")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.0/645.0 MB 796.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.6/258.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━

In [3]:
# fetching air quality data from open meteo
def get_city_coordinates(city_name):
    url = f"https://geocoding-api.open-meteo.com/v1/search?name={city_name}&count=1&format=json"
    response = requests.get(url)
    data = response.json()
    if "results" in data:
        return data["results"][0]["latitude"], data["results"][0]["longitude"], data["results"][0]["name"]
    else:
        raise ValueError(f"City '{city_name}' not found.")

lat, lon, city = get_city_coordinates(CITY)
print(f"Coordinates for {city}: Lat={lat}, Lon={lon}")

def fetch_aqi_data_openmeteo(lat, lon, days):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)

    # meteo's api endpoint
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "us_aqi,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust",
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "timezone": "auto"
    }

    print(f"Air Quality data from {start_date.date()} to {end_date.date()}...")
    response = requests.get(url, params=params)
    data = response.json()

    if "hourly" not in data:
        raise ValueError(f"Open-Meteo AQI API Error: {data}")

    df = pd.DataFrame({
        "timestamp": pd.to_datetime(data["hourly"]["time"]),
        "us_aqi": data["hourly"]["us_aqi"],
        "pm25": data["hourly"]["pm2_5"],
        "pm10": data["hourly"]["pm10"],
        "co": data["hourly"]["carbon_monoxide"],
        "no2": data["hourly"]["nitrogen_dioxide"],
        "so2": data["hourly"]["sulphur_dioxide"],
        "o3": data["hourly"]["ozone"],
        "dust": data["hourly"]["dust"]
    })

    return df

df_aqi = fetch_aqi_data_openmeteo(lat, lon, DAYS_TO_FETCH)
print(f"Fetched {len(df_aqi)} records.")
display(df_aqi.head(10))

Coordinates for Lahore: Lat=31.558, Lon=74.35071
Air Quality data from 2024-08-30 to 2026-07-31...
Fetched 16824 records.


,timestamp,us_aqi,pm25,pm10,co,no2,so2,o3,dust
0,2024-08-30 00:00:00,77,28.0,40.2,453.0,22.1,11.8,41.0,2.0
1,2024-08-30 01:00:00,77,26.5,38.3,407.0,18.2,12.1,40.0,3.0
2,2024-08-30 02:00:00,76,25.7,37.5,372.0,15.2,12.3,39.0,4.0
3,2024-08-30 03:00:00,76,25.4,37.6,353.0,13.7,12.6,37.0,5.0
4,2024-08-30 04:00:00,76,25.5,38.3,343.0,13.1,12.9,35.0,6.0
5,2024-08-30 05:00:00,76,22.9,35.3,286.0,10.6,12.5,48.0,8.0
6,2024-08-30 06:00:00,77,22.5,34.2,322.0,13.4,12.6,45.0,8.0
7,2024-08-30 07:00:00,77,23.7,35.9,371.0,17.0,12.7,42.0,8.0
8,2024-08-30 08:00:00,77,24.7,37.7,398.0,18.5,12.9,49.0,8.0
9,2024-08-30 09:00:00,77,24.6,37.6,383.0,15.7,13.4,73.0,8.0


In [4]:
# getting weather data for accurate prediction of next days as current air quality not enough to predict future.
def fetch_weather_data(lat, lon, days):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)

    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date.strftime("%Y-%m-%d"),
        "end_date": end_date.strftime("%Y-%m-%d"),
        "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,precipitation",
        "timezone": "auto"
    }

    print(f"Weather data from {start_date.date()} to {end_date.date()}...")
    response = requests.get(url, params=params)
    data = response.json()

    df = pd.DataFrame({
        "timestamp": pd.to_datetime(data["hourly"]["time"]),
        "temperature_c": data["hourly"]["temperature_2m"],
        "humidity_pct": data["hourly"]["relative_humidity_2m"],
        "wind_speed_kmh": data["hourly"]["wind_speed_10m"],
        "precipitation_mm": data["hourly"]["precipitation"]
    })

    return df

df_weather = fetch_weather_data(lat, lon, DAYS_TO_FETCH)

# merge aqi and weather df for 1 single df
df_raw = pd.merge(df_aqi, df_weather, on="timestamp", how="inner")
df_raw = df_raw.sort_values("timestamp").reset_index(drop=True)

print(f"Final raw dataset shape: {df_raw.shape}")
display(df_raw.head(10))

Weather data from 2024-08-30 to 2026-07-31...
Final raw dataset shape: (16824, 13)


,timestamp,us_aqi,pm25,pm10,co,no2,so2,o3,dust,temperature_c,humidity_pct,wind_speed_kmh,precipitation_mm
0,2024-08-30 00:00:00,77,28.0,40.2,453.0,22.1,11.8,41.0,2.0,24.5,87,13.2,0.0
1,2024-08-30 01:00:00,77,26.5,38.3,407.0,18.2,12.1,40.0,3.0,24.5,87,14.2,0.0
2,2024-08-30 02:00:00,76,25.7,37.5,372.0,15.2,12.3,39.0,4.0,24.6,86,14.3,0.0
3,2024-08-30 03:00:00,76,25.4,37.6,353.0,13.7,12.6,37.0,5.0,24.5,86,13.0,0.1
4,2024-08-30 04:00:00,76,25.5,38.3,343.0,13.1,12.9,35.0,6.0,24.5,86,14.5,0.0
5,2024-08-30 05:00:00,76,22.9,35.3,286.0,10.6,12.5,48.0,8.0,24.6,85,14.9,0.2
6,2024-08-30 06:00:00,77,22.5,34.2,322.0,13.4,12.6,45.0,8.0,24.7,94,10.0,0.3
7,2024-08-30 07:00:00,77,23.7,35.9,371.0,17.0,12.7,42.0,8.0,25.4,90,7.8,0.2
8,2024-08-30 08:00:00,77,24.7,37.7,398.0,18.5,12.9,49.0,8.0,26.0,87,13.8,0.1
9,2024-08-30 09:00:00,77,24.6,37.6,383.0,15.7,13.4,73.0,8.0,27.4,82,13.0,0.0


In [5]:
# feature engineerinh
# get features from timestamp to get periodic patterns
df_raw["hour"] = df_raw["timestamp"].dt.hour
df_raw["day"] = df_raw["timestamp"].dt.day
df_raw["month"] = df_raw["timestamp"].dt.month
df_raw["dayofweek"] = df_raw["timestamp"].dt.dayofweek  # 0=Monday - 6=Sunday

# reading of all pollutants for an hour and day ago repesented by lag (lagging)
df_raw["pm25_lag_1"] = df_raw["pm25"].shift(1)
df_raw["pm25_lag_24"] = df_raw["pm25"].shift(24)
df_raw["pm25_trend"] = df_raw["pm25_lag_1"] - df_raw["pm25_lag_24"]

df_raw["pm10_lag_1"] = df_raw["pm10"].shift(1)
df_raw["pm10_lag_24"] = df_raw["pm10"].shift(24)
df_raw["pm10_trend"] = df_raw["pm10_lag_1"] - df_raw["pm10_lag_24"]

df_raw["co_lag_1"] = df_raw["co"].shift(1)
df_raw["co_lag_24"] = df_raw["co"].shift(24)
df_raw["co_trend"] = df_raw["co_lag_1"] - df_raw["co_lag_24"]

df_raw["no2_lag_1"] = df_raw["no2"].shift(1)
df_raw["no2_lag_24"] = df_raw["no2"].shift(24)
df_raw["no2_trend"] = df_raw["no2_lag_1"] - df_raw["no2_lag_24"]

df_raw["so2_lag_1"] = df_raw["so2"].shift(1)
df_raw["so2_lag_24"] = df_raw["so2"].shift(24)
df_raw["so2_trend"] = df_raw["so2_lag_1"] - df_raw["so2_lag_24"]

df_raw["o3_lag_1"] = df_raw["o3"].shift(1)
df_raw["o3_lag_24"] = df_raw["o3"].shift(24)
df_raw["o3_trend"] = df_raw["o3_lag_1"] - df_raw["o3_lag_24"]

df_raw["dust_lag_1"] = df_raw["dust"].shift(1)
df_raw["dust_lag_24"] = df_raw["dust"].shift(24)
df_raw["dust_trend"] = df_raw["dust_lag_1"] - df_raw["dust_lag_24"]

df_raw["temperature_c_lag_1"] = df_raw["temperature_c"].shift(1)
df_raw["temperature_c_lag_24"] = df_raw["temperature_c"].shift(24)
df_raw["temperature_c_trend"] = df_raw["temperature_c_lag_1"] - df_raw["temperature_c_lag_24"]

df_raw["humidity_pct_lag_1"] = df_raw["humidity_pct"].shift(1)
df_raw["humidity_pct_lag_24"] = df_raw["humidity_pct"].shift(24)
df_raw["humidity_pct_trend"] = df_raw["humidity_pct_lag_1"] - df_raw["humidity_pct_lag_24"]

df_raw["wind_speed_kmh_lag_1"] = df_raw["wind_speed_kmh"].shift(1)
df_raw["wind_speed_kmh_lag_24"] = df_raw["wind_speed_kmh"].shift(24)
df_raw["wind_speed_kmh_trend"] = df_raw["wind_speed_kmh_lag_1"] - df_raw["wind_speed_kmh_lag_24"]

df_raw["precipitation_mm_lag_1"] = df_raw["precipitation_mm"].shift(1)
df_raw["precipitation_mm_lag_24"] = df_raw["precipitation_mm"].shift(24)
df_raw["precipitation_mm_trend"] = df_raw["precipitation_mm_lag_1"] - df_raw["precipitation_mm_lag_24"]



# rolling average
df_raw["pm25_rolling_24h"] = df_raw["pm25"].rolling(window=24, min_periods=1).mean()
df_raw["pm10_rolling_24h"] = df_raw["pm10"].rolling(window=24, min_periods=1).mean()
df_raw["co_rolling_24h"] = df_raw["co"].rolling(window=24, min_periods=1).mean()
df_raw["no2_rolling_24h"] = df_raw["no2"].rolling(window=24, min_periods=1).mean()
df_raw["so2_rolling_24h"] = df_raw["so2"].rolling(window=24, min_periods=1).mean()
df_raw["o3_rolling_24h"] = df_raw["o3"].rolling(window=24, min_periods=1).mean()
df_raw["dust_rolling_24h"] = df_raw["dust"].rolling(window=24, min_periods=1).mean()
df_raw["precipitation_mm_rolling_24h"] = df_raw["precipitation_mm"].rolling(window=24, min_periods=1).mean()
df_raw["wind_speed_kmh_rolling_24h"] = df_raw["wind_speed_kmh"].rolling(window=24, min_periods=1).mean()
df_raw["humidity_pct_rolling_24h"] = df_raw["humidity_pct"].rolling(window=24, min_periods=1).mean()
df_raw["temperature_c_rolling_24h"] = df_raw["temperature_c"].rolling(window=24, min_periods=1).mean()

# drop rows with nan entries for clean dataset
df_raw = df_raw.drop(columns=[
    "temperature_c",
    "humidity_pct",
    "wind_speed_kmh",
    "precipitation_mm",
    "pm25",
    "pm10",
    "co",
    "no2",
    "so2",
    "o3",
    "dust"
])
df_features = df_raw.dropna().reset_index(drop=True)

print(f"Original rows: {len(df_raw)} | Final  rows: {len(df_features)}")
print("New Dataset Columns:")
print(df_features.columns.tolist())
display(df_features.head(10))

Original rows: 16824 | Final  rows: 16800
New Dataset Columns:
['timestamp', 'us_aqi', 'hour', 'day', 'month', 'dayofweek', 'pm25_lag_1', 'pm25_lag_24', 'pm25_trend', 'pm10_lag_1', 'pm10_lag_24', 'pm10_trend', 'co_lag_1', 'co_lag_24', 'co_trend', 'no2_lag_1', 'no2_lag_24', 'no2_trend', 'so2_lag_1', 'so2_lag_24', 'so2_trend', 'o3_lag_1', 'o3_lag_24', 'o3_trend', 'dust_lag_1', 'dust_lag_24', 'dust_trend', 'temperature_c_lag_1', 'temperature_c_lag_24', 'temperature_c_trend', 'humidity_pct_lag_1', 'humidity_pct_lag_24', 'humidity_pct_trend', 'wind_speed_kmh_lag_1', 'wind_speed_kmh_lag_24', 'wind_speed_kmh_trend', 'precipitation_mm_lag_1', 'precipitation_mm_lag_24', 'precipitation_mm_trend', 'pm25_rolling_24h', 'pm10_rolling_24h', 'co_rolling_24h', 'no2_rolling_24h', 'so2_rolling_24h', 'o3_rolling_24h', 'dust_rolling_24h', 'precipitation_mm_rolling_24h', 'wind_speed_kmh_rolling_24h', 'humidity_pct_rolling_24h', 'temperature_c_rolling_24h']


,timestamp,us_aqi,hour,day,month,dayofweek,pm25_lag_1,pm25_lag_24,pm25_trend,pm10_lag_1,...,pm10_rolling_24h,co_rolling_24h,no2_rolling_24h,so2_rolling_24h,o3_rolling_24h,dust_rolling_24h,precipitation_mm_rolling_24h,wind_speed_kmh_rolling_24h,humidity_pct_rolling_24h,temperature_c_rolling_24h
0,2024-08-31 00:00:00,85,0,31,8,5,35.3,28.0,7.3,50.1,...,41.633333,378.000000,15.770833,12.612500,87.125000,6.333333,0.062500,9.483333,82.875000,27.300000
1,2024-08-31 01:00:00,86,1,31,8,5,34.6,26.5,8.1,49.0,...,42.141667,384.541667,16.250000,12.645833,87.083333,6.333333,0.062500,9.058333,83.166667,27.325000
2,2024-08-31 02:00:00,86,2,31,8,5,35.5,25.7,9.8,50.5,...,42.700000,391.458333,16.720833,12.658333,87.041667,6.291667,0.062500,8.595833,83.583333,27.333333
3,2024-08-31 03:00:00,87,3,31,8,5,35.6,25.4,10.2,50.9,...,43.258333,398.458333,17.162500,12.637500,86.958333,6.208333,0.058333,8.225000,84.000000,27.345833
4,2024-08-31 04:00:00,88,4,31,8,5,35.6,25.5,10.1,51.0,...,43.779167,405.250000,17.545833,12.583333,86.875000,6.125000,0.058333,7.812500,84.458333,27.354167
5,2024-08-31 05:00:00,89,5,31,8,5,35.4,22.9,12.5,50.8,...,44.187500,410.375000,17.891667,12.466667,86.416667,6.041667,0.050000,7.458333,84.958333,27.354167
6,2024-08-31 06:00:00,89,6,31,8,5,30.5,22.5,8.0,45.1,...,44.670833,415.875000,18.191667,12.375000,86.333333,6.041667,0.037500,7.404167,85.000000,27.370833
7,2024-08-31 07:00:00,90,7,31,8,5,31.1,23.7,7.4,45.8,...,45.270833,421.750000,18.429167,12.325000,86.791667,6.125000,0.029167,7.391667,85.083333,27.395833
8,2024-08-31 08:00:00,91,8,31,8,5,34.0,24.7,9.3,50.3,...,46.012500,427.458333,18.600000,12.316667,87.625000,6.250000,0.025000,7.158333,85.041667,27.458333
9,2024-08-31 09:00:00,92,9,31,8,5,36.3,24.6,11.7,55.5,...,47.016667,431.875000,18.700000,12.362500,88.708333,6.375000,0.025000,6.975000,84.833333,27.533333


In [ ]:
project = hopsworks.login(api_key_value=HOPSWORKS_API_KEY)

fs = project.get_feature_store()

feature_group = fs.get_or_create_feature_group(
    name="aqi_features",             # table name
    version=5,
    description="Hourly AQI, weather, and engineered features for ML training",
    primary_key=["timestamp"],
    event_time="timestamp",           # tells hopsworks this is time-series data
    time_travel_format="HUDI"
)

insert_response = feature_group.insert(df_features)


Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/42146


Uploading Dataframe: 100.00% |██████████| Rows 16800/16800 | Elapsed Time: 00:07 | Remaining Time: 00:00


Launching job: aqi_features_4_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://eu-west.cloud.hopsworks.ai:443/p/42146/jobs/named/aqi_features_4_offline_fg_materialization/executions


In [6]:
# project = hopsworks.login(api_key_value=HOPSWORKS_API_KEY)
# fs = project.get_feature_store()

# fg = fs.get_feature_group(name="aqi_features", version=2)

df_train = df_features
print(f"Fetched {len(df_train)} rows for training.")

# dropping time stamp and aqi
X = df_train.drop(columns=["timestamp", "us_aqi"])
y = df_train["us_aqi"]


# 80:20 split with 3 days data as real world simulation test
test_idx = len(X) - 72
split_idx = int(test_idx * 0.8)
X_train, X_val, X_test = X.iloc[:split_idx], X.iloc[split_idx:test_idx], X.iloc[test_idx:]
y_train, y_val, y_test = y.iloc[:split_idx], y.iloc[split_idx:test_idx], y.iloc[test_idx:]

print(f"Training set size: {len(X_train)} | Validation set size: {len(X_val)} | Test set size : {len(X_test)}")




Fetched 16800 rows for training.
Training set size: 13382 | Validation set size: 3346 | Test set size : 72


In [7]:

def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"{model_name} Evaluation:")
    print(f"   RMSE: {rmse:.2f}")
    print(f"   MAE:  {mae:.2f}")
    print(f"   R²:   {r2:.4f}")
    return [rmse, r2, mae] , model

In [9]:
print("Training xg boost model...")

# xgboost for a furthur

xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_eval, xgb_best = evaluate_model(xgb_model, X_val, y_val, "XGBoost")


Training xg boost model...
XGBoost Evaluation:
   RMSE: 10.45
   MAE:  5.59
   R²:   0.9350


In [13]:
print("Training lstm model...")

scaler_X = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_val_scaled = scaler_X.transform(X_val)
X_test_scaled = scaler_X.transform(X_test)


# Creating sequence for lstm
TIME_STEPS = 24

def create_sequences(X, y, time_steps):
    X_seq, y_seq = [], []

    for i in range(len(X) - time_steps):
        X_seq.append(X[i:i + time_steps])
        y_seq.append(y.iloc[i + time_steps])

    return np.array(X_seq), np.array(y_seq)

# Create sequences
X_train_seq, y_train_seq = create_sequences(
    X_train_scaled,
    y_train.reset_index(drop=True),
    TIME_STEPS
)

X_val_seq, y_val_seq = create_sequences(
    X_val_scaled,
    y_val.reset_index(drop=True),
    TIME_STEPS
)

X_test_seq, y_test_seq = create_sequences(
    X_test_scaled,
    y_test.reset_index(drop=True),
    TIME_STEPS
)
print(f"Training sequences   : {X_train_seq.shape}")
print(f"Validation sequences : {X_val_seq.shape}")
print(f"Test sequences       : {X_test_seq.shape}")

Training lstm model...
Training sequences   : (13358, 24, 48)
Validation sequences : (3322, 24, 48)
Test sequences       : (48, 24, 48)


In [14]:

# training model
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("Training LSTM Model...")

lstm_model = Sequential([
    LSTM(64, input_shape=(TIME_STEPS, X_train_seq.shape[2]), return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dense(16, activation='relu'),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')

lstm_model.fit(X_train_seq, y_train_seq, validation_split=0.2, epochs=50, batch_size=32, callbacks=[early_stop], verbose=0)
lstm_eval, lstm_best = evaluate_model(lstm_model, X_test_seq, y_test_seq, "LSTM")


Training LSTM Model...


2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 446ms/step
LSTM Evaluation:
   RMSE: 4.44
   MAE:  3.31
   R²:   0.9695


In [15]:
model_results = {
    "XGBoost": xgb_eval,
    "LSTM": lstm_eval
}

df_results = pd.DataFrame.from_dict(
    model_results,
    orient="index",
    columns=["RMSE", "R²", "MAE"]
)

# sort the grid by the best metric
df_results = df_results.sort_values(by="RMSE", ascending=True)

df_results["RMSE"] = df_results["RMSE"].map("{:.2f}".format)
df_results["MAE"] = df_results["MAE"].map("{:.2f}".format)
df_results["R²"] = df_results["R²"].map("{:.4f}".format)

print("MODEL PERFORMANCE LEADERBOARD (Sorted by lowest RMSE)")
display(df_results)


MODEL PERFORMANCE LEADERBOARD (Sorted by lowest RMSE)


,RMSE,R²,MAE
LSTM,4.44,0.9695,3.31
XGBoost,10.45,0.9350,5.59
